In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.utils import resample
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# -------------------- 1️ Load Data -------------------- #
df_orig = pd.read_csv("C:/Users/saira/OneDrive/Desktop/DePaul/DSC540/Project/final_cleaned_diabetic_data.csv")
df = df_orig.copy()  # To keep the original dataset unchanged


# Define features (X) and target (y)
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

# Train-Test Split (Stratified to balance class distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardization (Only for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Prepared: Train-Test Split Done!")

# -------------------- 2️ Feature Selection -------------------- 
xgb_selector = XGBClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=xgb_selector, n_features_to_select=20, step=1)
rfe.fit(X_train, y_train)

selected_features = X.columns[rfe.support_]
print(" Selected Features:", selected_features)

# Data with selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# -------------------- 3️ Cross-Validation -------------------- 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# -------------------- 4️ Model Training Function -------------------- 
def train_and_evaluate(model, name, X_train, X_test):
    print(f"\n Training {name} ")
    model.fit(X_train, y_train)
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics)
    print(f" {name} Cross-Validation Scores:")
    for metric in scoring_metrics:
        print(f"{metric.capitalize()}: {np.mean(cv_scores[f'test_{metric}']):.4f}")
    
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]
    print(f" {name} Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(f" {name} AUC-ROC Score: {roc_auc_score(y_test, y_probs):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# -------------------- 5️ Baseline Models (Before Tuning) -------------------- 
train_and_evaluate(LogisticRegression(solver="liblinear", class_weight="balanced"), "Logistic Regression", X_train_scaled, X_test_scaled)
train_and_evaluate(DecisionTreeClassifier(class_weight="balanced"), "Decision Tree", X_train, X_test)
train_and_evaluate(RandomForestClassifier(class_weight="balanced"), "Random Forest", X_train, X_test)
train_and_evaluate(XGBClassifier(scale_pos_weight=10), "XGBoost", X_train, X_test)

# -------------------- 6️ Tuned Models (With & Without Feature Selection) -------------------- 
tuned_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=9, min_samples_split=5, min_samples_leaf=3, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=3, class_weight="balanced_subsample", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=7, max_depth=9, learning_rate=0.03, colsample_bytree=0.8, subsample=0.8, random_state=42)
}

for name, model in tuned_models.items():
    train_and_evaluate(model, f"{name} (Tuned, All Features)", X_train, X_test)
    train_and_evaluate(model, f"{name} (Tuned, Selected Features)", X_train_selected, X_test_selected)

# -------------------- 7️ Ensemble Models -------------------- 
balanced_bagging = BalancedBaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=10, random_state=42),
    n_estimators=50, random_state=42)
train_and_evaluate(balanced_bagging, "Balanced Bagging", X_train, X_test)

ensemble_model = VotingClassifier(
    estimators=[
        ("rf", tuned_models["Random Forest"]),
        ("xgb", tuned_models["XGBoost"]),
        ("dt", tuned_models["Decision Tree"])
    ],
    voting="soft")
train_and_evaluate(ensemble_model, "Ensemble Model", X_train, X_test)

print("\n All Models Trained & Evaluated!")


Data Prepared: Train-Test Split Done!
 Selected Features: Index(['age', 'admission_type_id', 'discharge_disposition_id',
       'admission_source_id', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'number_diagnoses', 'change', 'diabetesMed',
       'ICDCat1', 'ICDCat2', 'ICDCat3', 'race_Caucasian', 'race_Hispanic',
       'max_glu_serum_None', 'A1Cresult_>8', 'A1Cresult_None',
       'num_visits_log', 'num_medications_log'],
      dtype='object')

 Training Logistic Regression 
 Logistic Regression Cross-Validation Scores:
Accuracy: 0.6225
Precision: 0.1620
Recall: 0.5545
F1: 0.2507
Roc_auc: 0.6325
 Logistic Regression Accuracy: 0.6285
 Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.64      0.75     17605
           1       0.17      0.56      0.26      2263

    accuracy                           0.63     19868
   macro avg       0.54      0.60      0.50     19868
weighted avg       0.83      0.63     

In [7]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# Apply SMOTE
smote = SMOTE(sampling_strategy=0.8, random_state=42)  
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Train the Random Forest Model with SMOTE data
rf_smote = RandomForestClassifier(
    n_estimators=300, 
    max_depth=15, 
    min_samples_split=5,  
    min_samples_leaf=3,  
    class_weight=None,  
    random_state=42
)

# Train and Evaluate
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_rf_smote = rf_smote.predict(X_test)
y_probs_rf_smote = rf_smote.predict_proba(X_test)[:, 1]

# Print Metrics
print("\n Random Forest (SMOTE) Accuracy:", accuracy_score(y_test, y_pred_rf_smote))
print("\n Classification Report (Random Forest with SMOTE):")
print(classification_report(y_test, y_pred_rf_smote, digits=2))

# AUC-ROC Score
auc_rf_smote = roc_auc_score(y_test, y_probs_rf_smote)
print(f"\n Random Forest (SMOTE) AUC-ROC Score: {auc_rf_smote:.4f}")

# Confusion Matrix
conf_matrix_rf_smote = confusion_matrix(y_test, y_pred_rf_smote)
print("\nConfusion Matrix (Random Forest with SMOTE):")
print(conf_matrix_rf_smote)



 Random Forest (SMOTE) Accuracy: 0.8313368230320113

 Classification Report (Random Forest with SMOTE):
              precision    recall  f1-score   support

           0       0.89      0.92      0.91     17605
           1       0.19      0.14      0.16      2263

    accuracy                           0.83     19868
   macro avg       0.54      0.53      0.53     19868
weighted avg       0.81      0.83      0.82     19868


 Random Forest (SMOTE) AUC-ROC Score: 0.6140

Confusion Matrix (Random Forest with SMOTE):
[[16197  1408]
 [ 1943   320]]


In [8]:
### ---------------- Logistic Regression without Class Imbalance --------------

In [9]:
train_and_evaluate(LogisticRegression(solver="liblinear"), "Logistic Regression", X_train_scaled, X_test_scaled)



 Training Logistic Regression 


C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_clas

 Logistic Regression Cross-Validation Scores:
Accuracy: 0.8861
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Roc_auc: 0.6320
 Logistic Regression Accuracy: 0.8861
 Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     17605
           1       0.00      0.00      0.00      2263

    accuracy                           0.89     19868
   macro avg       0.44      0.50      0.47     19868
weighted avg       0.79      0.89      0.83     19868

 Logistic Regression AUC-ROC Score: 0.6400
Confusion Matrix:
 [[17605     0]
 [ 2263     0]]


C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\saira\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
